# Graph, error and Genuine User diagnostics

Graph audits describe sampled native artefacts. Error families use their own
eligible and resolved cases. Genuine User uses six nominal categories and paired
transitions. These surfaces retain separate denominators and assessment revisions.

The default dataset is illustrative. Approved private scalar inputs use the same
manifest and functions in an ignored copy. No model requests are made.
See thesis Sections 2.6, 2.9, 3.3 and 3.5, and Appendix B.

In [2]:
import hashlib
import json
from pathlib import Path

from IPython.display import Markdown, display
from tabulate import tabulate

from rag_comparison import config
from rag_comparison.artifacts import read_jsonl, verify_artifacts
from rag_comparison.diagnostics import (
    graph_audit_summary,
    stratified_graph_sample,
    error_summary,
    question_property_memberships,
    genuine_user_counts,
    genuine_user_transitions,
)
from rag_comparison.artifacts import write_analysis_manifest

## Select bound diagnostic inputs

The graph population stores native IDs and strata. Review grades remain fixed.
Error records retain their dimension-specific applicability and resolution status.
Genuine User (GU) records identify their own accepted revision. A missing GU review
differs from an accepted `technical_failure` category.

In [3]:
data_dir = Path("../examples/evaluation")
scope = "PRIMARY"
output_dir = Path("../local/evaluation/diagnostic_example")
manifest = json.loads((data_dir / "manifest.json").read_text(encoding="utf-8"))
metadata = manifest["metadata"]
assert metadata["dataset_kind"] in ("illustrative", "accepted")
paths = verify_artifacts(data_dir, manifest)
questions = read_jsonl(paths["questions"])
population = read_jsonl(paths["graph_population"])
reviews = [row for row in read_jsonl(paths["graph_reviews"]) if row["scope"] == scope]
errors = [row for row in read_jsonl(paths["errors"]) if row["scope"] == scope]
gu_rows = read_jsonl(paths["gu_reviews"])
if metadata["dataset_kind"] == "accepted":
    assert data_dir.resolve().is_relative_to(Path("../local").resolve())
print(
    {
        "dataset_kind": metadata["dataset_kind"],
        "scope": scope,
        "GU_revision": metadata["review_revision_ids"]["GU"],
    }
)

{'dataset_kind': 'illustrative', 'scope': 'PRIMARY', 'GU_revision': 'EXAMPLE-REVIEW-GU'}


## Deterministic stratified graph selection

Selection draws up to 20 items per artefact type, without replacement. Each
nonempty stratum receives an item; remaining slots use proportional allocation
and largest remainders. Within strata, SHA256 of evaluation ID, type and stable ID
orders the items. Explicit ENTITY raw-type coverage can extend the entity sample
to one item per observed type. Configured types and observed labels are distinct.

The recorded sampling ID remains fixed across later assessment revisions. A result revision does not replace the selection seed.

A compact sampling input retains the original stratum sizes and only the bound hash prefixes needed for selection. The same allocation and ordering produce the same sample.

In [4]:
sample_condition = "G-ENTITY-POLICY-PROCESS-ROLE"
sample_population = [
    row
    for row in population
    if row["condition_id"] == sample_condition and row["artifact_type"] == "entities"
]
stratum_sizes = None
if "graph_strata" in paths:
    stratum_sizes = next(
        row["stratum_sizes"]
        for row in read_jsonl(paths["graph_strata"])
        if row["condition_id"] == sample_condition
        and row["artifact_type"] == "entities"
    )
sample = stratified_graph_sample(
    sample_population,
    metadata["graph_sampling_ids_by_condition"][sample_condition],
    "entities",
    entity_type_coverage=True,
    stratum_sizes=stratum_sizes,
)
assert len({row["stable_id"] for row in sample}) == len(sample)
assert {row["stratum"] for row in sample} == {
    row["stratum"] for row in sample_population
}
print(
    {
        "population": sum(stratum_sizes.values())
        if stratum_sizes
        else len(sample_population),
        "observed_raw_types": len({row["stratum"] for row in sample_population}),
        "sample": len(sample),
        "configured_types": len(config.EXTENDED_ENTITY_TYPES),
    }
)
display(
    Markdown(
        tabulate(
            [(row["stable_id"], row["stratum"]) for row in sample[:5]],
            headers=["Selected ID", "Observed stratum"],
            tablefmt="github",
        )
    )
)

{'population': 66, 'observed_raw_types': 22, 'sample': 22, 'configured_types': 9}


| Selected ID                                       | Observed stratum   |
|---------------------------------------------------|--------------------|
| EXAMPLE-G-ENTITY-POLICY-PROCESS-ROLE-entities-041 | raw-type-19        |
| EXAMPLE-G-ENTITY-POLICY-PROCESS-ROLE-entities-014 | raw-type-14        |
| EXAMPLE-G-ENTITY-POLICY-PROCESS-ROLE-entities-037 | raw-type-15        |
| EXAMPLE-G-ENTITY-POLICY-PROCESS-ROLE-entities-050 | raw-type-06        |
| EXAMPLE-G-ENTITY-POLICY-PROCESS-ROLE-entities-051 | raw-type-07        |

## Graph support and provenance have different denominators

Support excludes accepted `not_assessable` items. Provenance retains accepted
`unresolved` as a valid category; an unresolved review is instead missing judgement.
Contradiction and ambiguity remain separate flags. Graph summaries do not form a
single quality score or a constructed Classical-RAG comparison.

In [5]:
graph_results = {}
table = []
for cid in [
    key for key in metadata["conditions_by_scope"][scope] if key.startswith("G")
]:
    reuse = metadata.get("graph_audit_reuse", {}).get(cid)
    for artifact_type in config.GRAPH_AUDIT_ARTIFACT_TYPES:
        source = (
            reuse["source_condition"]
            if reuse and artifact_type in reuse["artifact_types"]
            else cid
        )
        selected = [
            row
            for row in reviews
            if row["condition_id"] == source and row["artifact_type"] == artifact_type
        ]
        if not selected:
            continue
        summary = graph_audit_summary(selected)
        graph_results[f"{cid}/{artifact_type}"] = {
            "review_source_condition": source,
            **summary,
        }
        support = summary["dimensions"]["support_status"]
        provenance = summary["dimensions"]["provenance_status"]
        table.append(
            (
                cid,
                artifact_type,
                source,
                summary["sample_N_total"],
                f"{support['counts']['supported']}/{support['k']}",
                f"{provenance['counts']['resolved']}/{provenance['k']}",
            )
        )
display(
    Markdown(
        tabulate(
            table,
            headers=[
                "Condition",
                "Artifact",
                "Review source",
                "Sample",
                "Supported / applicable",
                "Resolved / reviewed",
            ],
            tablefmt="github",
        )
    )
)

| Condition                    | Artifact            | Review source                |   Sample | Supported / applicable   | Resolved / reviewed   |
|------------------------------|---------------------|------------------------------|----------|--------------------------|-----------------------|
| G0                           | entities            | G0                           |       20 | 5/15                     | 7/20                  |
| G0                           | relationships       | G0                           |       20 | 5/15                     | 7/20                  |
| G0                           | community_reports   | G0                           |       20 | 5/15                     | 7/20                  |
| G0                           | local_search_traces | G0                           |       20 | 5/15                     | 7/20                  |
| G-CHUNK-STRUCT-800-0         | entities            | G-CHUNK-STRUCT-800-0         |       20 | 5/15                     | 7/20                  |
| G-CHUNK-STRUCT-800-0         | relationships       | G-CHUNK-STRUCT-800-0         |       20 | 5/15                     | 7/20                  |
| G-CHUNK-STRUCT-800-0         | community_reports   | G-CHUNK-STRUCT-800-0         |       20 | 5/15                     | 7/20                  |
| G-CHUNK-STRUCT-800-0         | local_search_traces | G-CHUNK-STRUCT-800-0         |       20 | 5/15                     | 7/20                  |
| G-EMBED-LARGE-1536           | entities            | G0                           |       20 | 5/15                     | 7/20                  |
| G-EMBED-LARGE-1536           | relationships       | G0                           |       20 | 5/15                     | 7/20                  |
| G-EMBED-LARGE-1536           | community_reports   | G0                           |       20 | 5/15                     | 7/20                  |
| G-ENTITY-POLICY-PROCESS-ROLE | entities            | G-ENTITY-POLICY-PROCESS-ROLE |       22 | 6/17                     | 8/22                  |
| G-ENTITY-POLICY-PROCESS-ROLE | relationships       | G-ENTITY-POLICY-PROCESS-ROLE |       20 | 5/15                     | 7/20                  |
| G-ENTITY-POLICY-PROCESS-ROLE | community_reports   | G-ENTITY-POLICY-PROCESS-ROLE |       20 | 5/15                     | 7/20                  |
| G-ENTITY-POLICY-PROCESS-ROLE | local_search_traces | G-ENTITY-POLICY-PROCESS-ROLE |       20 | 5/15                     | 7/20                  |

## Retain the source of inherited structural assessments

G-EMBED retains G0's structural graph records. G-FINAL retains G-ENTITY's structural
records, including its raw-type coverage, and has its own assessed Local Search
traces. The table identifies the actual review source. No trace judgement is
copied from another query context; absence remains visible in the input inventory.

## Count codes once per case on their own valid surface

Error codes are fixed diagnostic inputs. A lower quality grade does not create
an error code automatically. Each case contributes once to a code and once to
a family union, even when several member codes occur. Citation, content, retrieval,
projection and technical dimensions retain their own eligibility and unresolved cases.
Rates require an A surface with at least 20 valid cases; smaller or U surfaces retain counts.

In [6]:
error_results = {}
rows = []
groups = sorted(
    {(row["condition_id"], row["dimension"], row["cohort"]) for row in errors}
)
for cid, dimension, cohort in groups:
    selected = [
        row
        for row in errors
        if (row["condition_id"], row["dimension"], row["cohort"])
        == (cid, dimension, cohort)
    ]
    summary = error_summary(selected, cohort=cohort, families=config.ERROR_FAMILIES)
    error_results[f"{cid}/{dimension}/{cohort}"] = summary
    rows.append(
        (
            cid,
            dimension,
            cohort,
            summary["N_total"],
            summary["N_app"],
            summary["k_valid"],
            summary["N_unknown"],
            summary["known_with_error"],
        )
    )
display(
    Markdown(
        tabulate(
            rows[:8],
            headers=[
                "Condition",
                "Dimension",
                "Cohort",
                "N_total",
                "N_app",
                "Valid",
                "Unresolved",
                "With code",
            ],
            tablefmt="github",
        )
    )
)

| Condition            | Dimension   | Cohort   |   N_total |   N_app |   Valid |   Unresolved |   With code |
|----------------------|-------------|----------|-----------|---------|---------|--------------|-------------|
| C-CHUNK-STRUCT-800-0 | answer      | A        |        24 |      24 |      24 |            0 |           4 |
| C-CHUNK-STRUCT-800-0 | answer      | U        |         4 |       4 |       4 |            0 |           0 |
| C-CHUNK-STRUCT-800-0 | citation    | A        |        24 |      24 |      24 |            0 |           4 |
| C-CHUNK-STRUCT-800-0 | citation    | U        |         4 |       1 |       1 |            0 |           0 |
| C-CHUNK-STRUCT-800-0 | retrieval   | A        |        24 |      24 |      24 |            0 |           0 |
| C-CHUNK-STRUCT-800-0 | retrieval   | U        |         4 |       0 |       0 |            0 |           0 |
| C-CHUNK-STRUCT-800-0 | technical   | A        |        24 |      24 |      24 |            0 |           0 |
| C-CHUNK-STRUCT-800-0 | technical   | U        |         4 |       4 |       4 |            0 |           0 |

## Question properties can overlap

Numerical, multi-evidence and exception-sensitive properties describe overlapping
A-question groups. Their counts are not disjoint totals. Any outcome comparison
within a group still needs that outcome's applicable, complete question pairs.

In [7]:
properties = question_property_memberships(questions)
display(
    Markdown(
        tabulate(
            [(name, len(ids)) for name, ids in properties.items()],
            headers=["Property", "A questions"],
            tablefmt="github",
        )
    )
)
overlap = set(properties["Numerical"]) & set(properties["Exception-sensitive"])
print({"numerical_and_exception_sensitive": len(overlap)})

| Property            |   A questions |
|---------------------|---------------|
| Numerical           |             6 |
| Multi-evidence      |             8 |
| Exception-sensitive |             5 |

{'numerical_and_exception_sensitive': 2}


## Genuine User uses nominal categories

The four systems use the same separate user questions and accepted references.
Six categories describe the complete answer decision. They have no ordinal
ordering and are not converted to PRIMARY correctness grades or dominance signs.
An accepted technical-failure category is counted; an absent assessment stays missing.

In [8]:
gu_ids = sorted({row["question_id"] for row in gu_rows})
gu_conditions = ("C0", "G0", "C-FINAL", "G-FINAL")
gu_values = {
    cid: {
        row["question_id"]: row["category"]
        for row in gu_rows
        if row["condition_id"] == cid and row["review_status"] == "reviewed"
    }
    for cid in gu_conditions
}
gu_results = {
    cid: genuine_user_counts(values, gu_ids) for cid, values in gu_values.items()
}
display(
    Markdown(
        tabulate(
            [
                (cid, result["N_total"], result["k"], *result["categories"].values())
                for cid, result in gu_results.items()
            ],
            headers=["Condition", "N_total", "Reviewed", *config.GU_CATEGORIES],
            tablefmt="github",
        )
    )
)

| Condition   |   N_total |   Reviewed |   correct |   partially_correct |   incorrect |   appropriate_abstention |   inappropriate_abstention |   technical_failure |
|-------------|-----------|------------|-----------|---------------------|-------------|--------------------------|----------------------------|---------------------|
| C0          |        24 |         24 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |
| G0          |        24 |         24 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |
| C-FINAL     |        24 |         24 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |
| G-FINAL     |        24 |         24 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |

## Match category transitions question by question

Transitions preserve their left and right category names. The same-question
intersection determines the paired denominator. These tables show movements
between nominal categories, without a combined performance score.

In [9]:
gu_pairs = {}
for left, right in (
    ("C0", "G0"),
    ("C0", "C-FINAL"),
    ("G0", "G-FINAL"),
    ("C-FINAL", "G-FINAL"),
):
    gu_pairs[f"{left}/{right}"] = genuine_user_transitions(
        gu_values[left], gu_values[right], gu_ids
    )
selected = gu_pairs["C0/G0"]
print(
    {
        "paired_questions": selected["common_k"],
        "missing_pairs": selected["missing_pairs"],
    }
)
display(
    Markdown(
        tabulate(
            [
                (row["left_category"], row["right_category"], row["count"])
                for row in selected["category_transitions"]
            ],
            headers=["C0 category", "G0 category", "Questions"],
            tablefmt="github",
        )
    )
)

{'paired_questions': 24, 'missing_pairs': 0}


| C0 category              | G0 category              |   Questions |
|--------------------------|--------------------------|-------------|
| appropriate_abstention   | inappropriate_abstention |           4 |
| correct                  | partially_correct        |           4 |
| inappropriate_abstention | technical_failure        |           4 |
| incorrect                | appropriate_abstention   |           4 |
| partially_correct        | incorrect                |           4 |
| technical_failure        | correct                  |           4 |

## Save diagnostic results with their input identity

The recalculation output retains graph review sources, separate diagnostic bases,
property memberships and complete nominal transitions. Private executions and
outputs stay under ignored `local/`; export never requires new model requests.

In [10]:
assert output_dir.resolve().is_relative_to(Path("../local").resolve())
output_dir.mkdir(parents=True, exist_ok=True)
result = {
    "scope": scope,
    "dataset_kind": metadata["dataset_kind"],
    "input_manifest_sha256": hashlib.sha256(
        (data_dir / "manifest.json").read_bytes()
    ).hexdigest(),
    "graph": graph_results,
    "errors": error_results,
    "properties": properties,
    "gu_counts": gu_results,
    "gu_transitions": gu_pairs,
}
(output_dir / "diagnostic_analysis.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print({"output_file": "diagnostic_analysis.json", "model_requests": 0})
write_analysis_manifest(
    output_dir,
    data_dir / "manifest.json",
    {"diagnostic_analysis": "diagnostic_analysis.json"},
)

{'output_file': 'diagnostic_analysis.json', 'model_requests': 0}


{'schema_version': 'evaluation-outputs-v1',
 'input_manifest_sha256': '4e5bcd07361c4937ab889f1d820febf8958e3ef4bd6ad6c17f4b5f5fc2e4e566',
 'artifacts': {'diagnostic_analysis': {'path': 'diagnostic_analysis.json',
   'sha256': 'c89640147e359ffe2c0b7a6e77cf66a7c7a192778d5c1704bdaf381c9d9465ef'}}}